In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("../data/UNSW_NB15_training-set.csv")
test = pd.read_csv("../data/UNSW_NB15_testing-set.csv")

In [3]:
train.shape, test.shape

((175341, 45), (82332, 45))

In [4]:
train.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [5]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 175341 non-null  int64  
 1   dur                175341 non-null  float64
 2   proto              175341 non-null  str    
 3   service            175341 non-null  str    
 4   state              175341 non-null  str    
 5   spkts              175341 non-null  int64  
 6   dpkts              175341 non-null  int64  
 7   sbytes             175341 non-null  int64  
 8   dbytes             175341 non-null  int64  
 9   rate               175341 non-null  float64
 10  sttl               175341 non-null  int64  
 11  dttl               175341 non-null  int64  
 12  sload              175341 non-null  float64
 13  dload              175341 non-null  float64
 14  sloss              175341 non-null  int64  
 15  dloss              175341 non-null  int64  
 16  sinpkt       

In [6]:
features = pd.read_csv("../data/NUSW-NB15_features.csv", encoding="latin-1")
pd.set_option("display.max_colwidth", None)
features

,No.,Name,Type,Description
0,1,srcip,nominal,Source IP address
1,2,sport,integer,Source port number
2,3,dstip,nominal,Destination IP address
3,4,dsport,integer,Destination port number
4,5,proto,nominal,Transaction protocol
5,6,state,nominal,"Indicates to the state and its dependent protocol, e.g. ACC, CLO, CON, ECO, ECR, FIN, INT, MAS, PAR, REQ, RST, TST, TXD, URH, URN, and (-) (if not used state)"
6,7,dur,Float,Record total duration
7,8,sbytes,Integer,Source to destination transaction bytes
8,9,dbytes,Integer,Destination to source transaction bytes
9,10,sttl,Integer,Source to destination time to live value


In [7]:
train["label"].value_counts(normalize=True)

label
1    0.680622
0    0.319378
Name: proportion, dtype: float64

In [8]:
train["attack_cat"].value_counts()

attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64

In [9]:
train[["proto", "service", "state"]].nunique()

proto      133
service     13
state        9
dtype: int64

In [10]:
pd.crosstab(train["proto"], train["label"]).sort_values(0, ascending=False).head(15)

label,0,1
proto,,
tcp,39121,40825
udp,13922,49361
arp,2859,0
ospf,64,2531
igmp,18,0
icmp,15,0
rtp,1,0
argus,0,98
a/n,0,100


In [11]:
train.drop(columns="id").duplicated().sum()

np.int64(67601)

In [12]:
dupes = train.drop(columns="id").duplicated()
train[dupes]["attack_cat"].value_counts()

attack_cat
Generic           35819
Exploits          13549
DoS                8458
Normal             4110
Reconnaissance     2969
Fuzzers            2034
Analysis            406
Backdoor            211
Shellcode            42
Worms                 3
Name: count, dtype: int64

In [13]:
pd.merge(train.drop(columns="id").drop_duplicates(),
         test.drop(columns="id").drop_duplicates()).shape

(940, 44)

In [14]:
train.drop(columns=["id", "label", "attack_cat"]).duplicated().sum()

np.int64(74301)

In [15]:
train.groupby("label")[["sttl", "dttl", "ct_state_ttl", "sbytes", "dur"]].median()

,sttl,dttl,ct_state_ttl,sbytes,dur
label,,,,,
0,31.0,29.0,0.0,1470.0,0.038602
1,254.0,0.0,2.0,200.0,0.000009


In [16]:
pd.crosstab(train["sttl"], train["label"])

label,0,1
sttl,,
0,2859,303
1,64,0
29,2,0
31,39455,0
60,6,0
62,2167,15514
63,32,0
64,181,0
252,2,0


In [17]:
pd.crosstab(train["ct_state_ttl"], train["label"])

label,0,1
ct_state_ttl,,
0,39907,884
1,8705,40650
2,5504,76550
3,1019,192
6,865,1065


## Phase 1 findings

**Dataset:** UNSW-NB15 partitioned sets: 175,341 training rows, 82,332 test rows, 45 columns.
Targets are `label` (0 = normal, 1 = attack) and `attack_cat` (Normal + 9 attack categories);
`id` is a row number. None of these three can be model inputs.

### Class balance
- Training set: 119,341 attack (68%) vs 56,000 normal (32%). Attacks are the majority,
  unlike a real network, where they'd be rare.
- Smallest attack categories (Worms, Shellcode, Backdoor, Analysis) have very few rows,
  so they will be the hardest to learn in Phase 3.

### Protocols (`proto`)
- 133 unique protocols. Normal traffic uses only 7: tcp, udp, arp, ospf, igmp, icmp, rtp.
- The other 126 appear only in attack traffic, almost all with ~100 rows each.
  This looks like generated traffic (a tool cycling through protocol numbers), not real attacker behavior.
- Risk: the model can learn "unusual protocol = attack" as a shortcut.
- Plan: group rare protocols into "other" in Phase 2; test removing `proto` in Phase 4.

### Duplicates and leakage
- 67,601 of 175,341 training rows (39%) are exact duplicates (ignoring `id`),
  mostly Generic (35,819), Exploits (13,549) and DoS (8,458). Only 4,110 are Normal.
- Plain cross-validation would put copies of a row in both training and validation folds,
  which inflates scores. Plan: drop duplicates from the training set before cross-validation.
- 940 unique rows appear in both training and test (~1% of test). Minor leakage; note it in results.
- 6,700 rows have identical features but different labels. No model can classify all of these correctly.

### TTL artifact
- `sttl` of 62 or 254 covers 99.7% of attacks (119,027 of 119,341).
- `sttl` = 31 appears in 39,455 normal rows and never in attacks.
- `sttl` = 0 matches the 2,859 ARP rows (ARP has no IP TTL).
- TTL reflects which testbed hosts sent the traffic (initial TTL 255 vs 64 vs 32), not attack behavior.
  `ct_state_ttl` is derived from TTL and inherits the same artifact.
- Rule baseline "attack if `sttl` in {62, 254}": 99.7% recall, ~24% false positive rate
  (13,397 of 56,000 normal flagged).
- Plan: use this rule as the Phase 2 baseline; remove `sttl`, `dttl` and `ct_state_ttl` in the Phase 4 ablation.

### Other notes
- The feature description file ships with a typo in its name (`NUSW-NB15_features.csv`)
  and needs `encoding="latin-1"` to load.

### Carry into Phase 2
1. Drop `id`; keep `label` as the target.
2. Drop duplicate rows from the training set before cross-validation.
3. Group rare `proto` values into "other".
4. Beat the TTL rule baseline, mainly on false positive rate.
5. Use the test set only once, at the end.